# Week 2 — Statistics I: Core Concepts through ANOVA

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

# Reproducible random-number generator
rng = np.random.default_rng(3060)

### Example dataset: neuronal firing rate

We will use one small, synthetic neuroscience dataset throughout the notebook so that the statistics remain connected to one experimental question.

Imagine an experiment measuring mean firing rate in neurons from animals receiving one of three treatments:

- `control`
- `low`
- `high`

We also record `sex`, which gives us a second experimental factor for the two-way ANOVA example later.

In [ ]:
# Create a small balanced teaching dataset: 12 observations per treatment × sex cell
means = {
    ('control', 'female'): 8.0,
    ('low',     'female'): 9.0,
    ('high',    'female'): 9.5,
    ('control', 'male'):   8.5,
    ('low',     'male'):  10.2,
    ('high',    'male'):  12.5,
}

rows = []
for treatment in ['control', 'low', 'high']:
    for sex in ['female', 'male']:
        values = rng.normal(loc=means[(treatment, sex)], scale=1.0, size=12)
        for value in values:
            rows.append({
                'treatment': treatment,
                'sex': sex,
                'firing_rate_hz': value
            })

df = pd.DataFrame(rows)
df.head()

## 1. Describe before you test

Before running a statistical test, ask:

1. What is my outcome variable?
2. What groups or conditions am I comparing?
3. How many observations are in each group?
4. What does the distribution look like?
5. Are there obvious outliers or data-quality problems?

### A few quantities worth keeping straight

From the earlier biostatistics course, there are three related ideas that are useful to distinguish:

**Mean** describes the center of a sample:

$$
\bar{X} = \frac{\sum X}{n}
$$

**Variance and standard deviation** describe how spread out individual observations are around the mean:

$$
s^2 = \frac{\sum (X-\bar{X})^2}{n-1}
\qquad
s = \sqrt{s^2}
$$

**Standard error of the mean (SEM)** describes uncertainty in the estimated mean rather than variability among individual observations:

$$
s_{\bar{X}} = \frac{s}{\sqrt{n}}
$$

Notice that the SEM becomes smaller as sample size increases. The observations themselves do not necessarily become less variable; our estimate of the mean becomes more precise.

#### Normal distributions and sample means

A normal (Gaussian) distribution is characterized by its mean and standard deviation. One reason the normal distribution appears so often in statistics is the **Central Limit Theorem**: under broad conditions, the distribution of sample means becomes approximately normal as sample size increases, even when the original observations are not perfectly normally distributed.

For this course, we will usually let `pandas`, NumPy, and SciPy calculate these quantities. The important part is understanding what each quantity tells us about the data.


In [ ]:
# A compact summary by treatment
summary = df.groupby('treatment', observed=True)['firing_rate_hz'].describe()
summary

In [ ]:
# Visualize the same outcome before testing it
ax = df.boxplot(column='firing_rate_hz', by='treatment', grid=False)
plt.suptitle('')
plt.title('Firing rate by treatment')
plt.xlabel('Treatment')
plt.ylabel('Firing rate (Hz)')
plt.show()

## 2. Hypothesis testing: the basic logic

We conduct a statistical analysis to assess a hypothesis.

For a group comparison:

- **Null hypothesis ($H_0$):** there is no systematic difference between the populations represented by the groups; observed differences are compatible with sampling variability.
- **Alternative hypothesis ($H_A$):** there is a systematic difference between the populations represented by the groups.

A statistical test combines:

> **observed difference or structure** ÷ **expected variability/uncertainty**

The resulting test statistic is then related to a sampling distribution to obtain a **p-value**.

### Interpreting a p-value
A p-value asks:

> If the null hypothesis and the test's assumptions were true, how surprising would a result at least this extreme be?

A small p-value can provide evidence against the null hypothesis. It does **not** tell us:

- the probability that the null hypothesis is true,
- whether an effect is biologically important,
- whether the experiment was well designed,
- whether assumptions are reasonable,
- whether the finding will replicate.

### Parametric and nonparametric approaches

In the original biostatistics course, we distinguished two broad families of statistical procedures:

- **Parametric statistics** use a model described by a set of parameters. For example, a normal distribution can be described by its mean and standard deviation.
- **Nonparametric statistics** make fewer distributional assumptions and often work with ranks, frequencies, or other features of the data rather than the same mean/variance model.

This distinction is useful, but it is **not** a rule that automatically maps “normal data” to one test and “non-normal data” to another. Experimental design, independence, outliers, sample size, and the scientific question also matter. We will return to assumptions and test selection in Week 3.

### Significance level versus p-value

Before looking at the data, we may choose a significance level such as $\alpha=0.05$. The **p-value** is calculated from the observed data. We compare the p-value with $\alpha$ when deciding whether the data provide enough evidence to reject the null hypothesis.

If we fail to reject $H_0$, that does **not** prove that the groups are identical. It means that, with the data and analysis we have, we do not have enough evidence to conclude that they differ.


## 3. Comparing two independent groups

### Example question
Do firing rates differ between the `control` and `high` treatment groups?

For two independent groups, the t-statistic can be understood as:

$$
 t \approx \frac{\text{difference in group means}}{\text{uncertainty in that difference}}
$$

In [ ]:
control = df.loc[df['treatment'] == 'control', 'firing_rate_hz']
high = df.loc[df['treatment'] == 'high', 'firing_rate_hz']

print('Control mean:', control.mean())
print('High mean:   ', high.mean())
print('Difference:  ', high.mean() - control.mean())

### Independent-samples t-test

The **t distribution** is related to the standard normal distribution but has heavier tails. Its shape depends on the degrees of freedom; as the degrees of freedom increase, it approaches the standard normal distribution.

There are several forms of the t-test:

- one-sample t-test
- two-sample (independent) t-test
- paired t-test

A t-test can also be one-tailed or two-tailed depending on the hypothesis. In this notebook we use a **two-sided, two-sample comparison**.

The core idea is that the difference in sample means is interpreted relative to the uncertainty in that difference:

$$
t = \frac{\bar{X}_1-\bar{X}_2}{s_{\bar{X}_1-\bar{X}_2}}
$$

For two independent groups, the standard error of the difference depends on the variability and sample size in **both** groups:

$$
s_{\bar{X}_1-\bar{X}_2}
= \sqrt{\frac{s_1^2}{n_1}+\frac{s_2^2}{n_2}}
$$

The SciPy call below uses Welch's version of the independent t-test (`equal_var=False`), so we do not have to assume that the two groups have exactly equal variances.


In [ ]:
t_result = stats.ttest_ind(control, high, equal_var=False)
t_result

### Nonparametric alternative: Mann–Whitney U
A common nonparametric alternative for **two independent groups** is the Mann–Whitney U test.

It works with ranks rather than using the same mean/variance model as a t-test.

> Do not reduce the distinction to “normal = t-test, non-normal = Mann–Whitney.” Test choice also depends on the scientific question, study design, distribution shape, outliers, sample size, and what parameter you want to infer about.

In [ ]:
u_result = stats.mannwhitneyu(control, high, alternative='two-sided')
u_result

### Paired data are different

If the **same animal, subject, slice, or cell** is measured twice, the observations are paired rather than independent.

| Design | Parametric example | Nonparametric example |
|---|---|---|
| Two independent groups | `stats.ttest_ind()` | `stats.mannwhitneyu()` |
| Two paired measurements | `stats.ttest_rel()` | `stats.wilcoxon()` |

The experimental design determines which row is appropriate.

## 4. Comparing more than two groups: one-way ANOVA

### Example question
Does firing rate differ among `control`, `low`, and `high` treatment groups?

The key intuition from the earlier ANOVA notebooks is:

$$
F = \frac{\text{variability between groups}}{\text{variability within groups}}
$$

- If group means are close relative to within-group variability, $F$ tends to be smaller.
- If group means are far apart relative to within-group variability, $F$ tends to be larger.

### Where the ANOVA quantities come from

- **SSB** — sum of squares **between** groups: how far the group means are from the overall mean.
- **SSE** — sum of squares **within** groups (error): how far individual observations are from their own group mean.
- **MSB** — mean square between groups: SSB divided by its degrees of freedom.
- **MSE** — mean square error: SSE divided by its degrees of freedom.

The F statistic is then

$$
F = \frac{MSB}{MSE}
$$

If the differences among group means are large compared with the variability of observations within groups, the F statistic becomes larger.

In [ ]:
groups = [
    df.loc[df['treatment'] == treatment, 'firing_rate_hz']
    for treatment in ['control', 'low', 'high']
]

anova_result = stats.f_oneway(*groups)
anova_result

### What does a significant one-way ANOVA tell us?

The null hypothesis is that the population means are equal across the groups included in the model.

If the ANOVA is significant, the conclusion is:

> **At least one group differs.**

It does **not** tell us which specific pairs differ.

#### What comes after ANOVA? Post-hoc and multiple comparisons

When we test many hypotheses, the chance of obtaining at least one false positive increases.


Methods such as **Bonferroni** or **Holm** adjust for this multiple-comparison problem. We will save the details for Week 4, where multiple comparisons and false discovery rate are covered explicitly.


### Nonparametric alternative: Kruskal–Wallis

In [ ]:
kruskal_result = stats.kruskal(*groups)
kruskal_result

## 5. Two factors: two-way ANOVA

Our dataset contains two factors:

- `treatment`: control, low, high
- `sex`: female, male

A two-way ANOVA can address three questions:

1. **Main effect of treatment:** Are outcomes different across treatment levels, averaging across sex?
2. **Main effect of sex:** Are outcomes different across sex, averaging across treatment?
3. **Treatment × sex interaction:** Does the effect of treatment depend on sex?

In [ ]:
# First, inspect group means
cell_means = df.groupby(['treatment', 'sex'], observed=True)['firing_rate_hz'].mean().unstack()
cell_means

In [ ]:
# Plot the cell means to make an interaction easier to see
ordered = ['control', 'low', 'high']
means_for_plot = cell_means.loc[ordered]

for sex in means_for_plot.columns:
    plt.plot(ordered, means_for_plot[sex], marker='o', label=sex)

plt.xlabel('Treatment')
plt.ylabel('Mean firing rate (Hz)')
plt.title('Treatment × sex')
plt.legend(title='Sex')
plt.show()

In [ ]:
# Fit a model with both main effects and their interaction
model = smf.ols('firing_rate_hz ~ C(treatment) * C(sex)', data=df).fit()

# Balanced teaching dataset: Type II ANOVA table is sufficient for this introduction
anova_2way = anova_lm(model, typ=2)
anova_2way

### Read the ANOVA table conceptually

Look for these rows:

- `C(treatment)` → treatment main effect
- `C(sex)` → sex main effect
- `C(treatment):C(sex)` → interaction

**Question:** Based on the interaction plot and ANOVA table, does the treatment effect look the same for both sexes?

## 6. A practical test-selection framework

Do not start with the name of a statistical test. Start with the design.

| Experimental question / design | A reasonable starting point |
|---|---|
| One numerical outcome, two independent groups | Independent t-test; consider Mann–Whitney U |
| One numerical outcome, same subjects measured twice | Paired t-test; consider Wilcoxon signed-rank |
| One numerical outcome, 3+ independent groups, one factor | One-way ANOVA; consider Kruskal–Wallis |
| One numerical outcome, two categorical factors | Two-way ANOVA / regression model with interaction |

Then ask:

1. **What is the experimental unit?**
2. **Are observations independent or paired/repeated?**
3. **How many factors and levels are present?**
4. **What does the raw data look like?**
5. **What assumptions does the model make?**
6. **What effect, uncertainty, and limitations should be reported?**
